# Model Building

Compare Random Forest, Gradient Boosting, and XGBoost on a 1,000-row training subset, report Train and Validation AUC-ROC for each, then pick the best model by validation AUC-ROC and train-validation gap.

In [9]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

data_dir = Path("../data/processed")

train_df = pd.read_csv(data_dir / "train_feature_engineered.csv")
validation_df = pd.read_csv(data_dir / "validation_feature_engineered.csv")
train_df = train_df.sample(n=1000, random_state=42)

target = "clicked"

X_train = pd.get_dummies(train_df.drop(columns=[target]), dummy_na=True)
y_train = train_df[target]

X_validation = pd.get_dummies(validation_df.drop(columns=[target]), dummy_na=True)
y_validation = validation_df[target]

X_train.columns = X_train.columns.astype(str).str.replace(r"[\[\]<>]", "", regex=True)
X_validation.columns = X_validation.columns.astype(str).str.replace(r"[\[\]<>]", "", regex=True)

X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0)

X_train.shape, X_validation.shape

((1000, 48), (67500, 48))

In [10]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count if positive_count else 1.0

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    ),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        random_state=42,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
    ),
}

results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)

    train_proba = model.predict_proba(X_train)[:, 1]
    validation_proba = model.predict_proba(X_validation)[:, 1]

    train_auc = roc_auc_score(y_train, train_proba)
    validation_auc = roc_auc_score(y_validation, validation_proba)

    results.append(
        {
            "model": model_name,
            "train_auc": train_auc,
            "validation_auc": validation_auc,
            "gap": train_auc - validation_auc,
        }
    )

results_df = pd.DataFrame(results).sort_values(
    ["validation_auc", "gap"], ascending=[False, True]
).reset_index(drop=True)
best_row = results_df.iloc[0]

print("Best model by validation AUC-ROC, then smallest train-validation gap:")
print(best_row[["model", "train_auc", "validation_auc", "gap"]])
results_df

Best model by validation AUC-ROC, then smallest train-validation gap:
model             Gradient Boosting
train_auc                  0.999716
validation_auc             0.683411
gap                        0.316305
Name: 0, dtype: object


,model,train_auc,validation_auc,gap
0,Gradient Boosting,0.999716,0.683411,0.316305
1,Random Forest,1.000000,0.672083,0.327917
2,XGBoost,1.000000,0.657004,0.342996


## Summary

- Compared `RandomForestClassifier`, `GradientBoostingClassifier`, and `XGBClassifier` on the same 1,000-row training subset.
- Reported Train and Validation AUC-ROC for each model.
- Ranked the models by validation AUC-ROC and used the train-validation gap as the tie-breaker.